# Qwen 3.5 - 0.8B Model implementation from Scratch

## Positional Embedding and RMSNorm Helper function

In [1]:
import torch, math

def rope_rotate(head_dim, context_length, device='cpu', theta=1_000_000):
    """
    Precompute RoPE cos/sin tables.
    head_dim : dimension being rotated (pass rope_dim for partial RoPE)
    theta    : RoPE base frequency (1_000_000 for Qwen3; 10_000_000 for Qwen3.5)
    Returns cos, sin of shape (1, 1, context_length, head_dim).
    """

    half = head_dim // 2
    # Generate position indices
    positions = torch.arange(context_length, dtype=torch.float32, device=device)
    freqs = torch.exp(-math.log(theta) * torch.arange(0, half, device=device) / half)  # (half,)
    angles = torch.einsum('l, h -> lh', positions.float(), freqs)                  # (L, half)

    combined_angles = torch.cat([angles, angles], dim=1)                         # (L, head_dim)

    cos = combined_angles.cos()[None, None, :, :]                                        # (1,1,L,head_dim)

    sin = combined_angles.sin()[None, None, :, :]                                          # (1,1,L,head_dim)

    return cos, sin


def apply_rope(x, cos, sin):
    """
    x: (B, H, L, Dh) queries or keys
    positions: (L,) absolute positions (0..L-1)
    """
    B, H, L, Dh = x.shape
    half = Dh // 2


    x_upper = x[..., :half]
    x_lower = x[..., half:]

    x_bar = torch.cat((-x_lower, x_upper), dim=-1)              # (B,H,L,Dh)

    cos = cos[:,:,:L, :]                                               # (1,1,L,Dh)
    sin = sin[:,:,:L, :]                                                # (1,1,L,Dh)

    x_rot = (x * cos) + (x_bar * sin)
    return x_rot.to(x.dtype)


def test_apply_rope():
    x = torch.randn(2, 8, 512, 128)
    cos, sin = rope_rotate(128, 512)
    x_out = apply_rope(x, cos, sin)
    assert x_out.shape == (2, 8, 512, 128)


# if __name__ == '__main__':
#     # test_apply_rope()
#     pass

In [2]:
from torch import nn
import torch
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, n_embed, eps=1e-6):
        super().__init__()
        # Qwen3.5 uses zero-init weight with (1 + weight) scaling
        self.weight = nn.Parameter(torch.zeros(n_embed))
        self.variance_epsilon = eps

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.variance_epsilon)

    def forward(self, x):
        input_dtype = x.dtype
        x_norm = self._norm(x.float())
        x_norm = x_norm * (1.0 + self.weight.float())
        return x_norm.to(input_dtype)


class RMSNormGated(nn.Module):
    """
    RMSNorm followed by an element-wise SiLU gate.
    Matches Qwen3_5RMSNormGated from the HF implementation.
    forward(x, gate) → scale * rms_norm(x) * silu(gate)  (all cast to input dtype)
    """
    def __init__(self, n_embed, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embed))
        self.variance_epsilon = eps

    def forward(self, x, gate):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        x = self.weight * x.to(input_dtype)
        return (x * F.silu(gate.to(torch.float32))).to(input_dtype)

In [3]:
import torch



def model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024 ** 3)

    return total_memory_gb

## Group Query Full Attention

In [4]:
import torch
import torch.nn.functional as F
from einops import rearrange
from torch import nn, einsum


def l2norm(x, dim=-1, eps=1e-6):
    """Unit L2 normalisation without a learnable scale (matches FLA / HF convention)."""
    return x * torch.rsqrt((x * x).sum(dim=dim, keepdim=True) + eps)


class GQAttention(nn.Module):
    """
    Grouped Query Attention with partial RoPE and optional sigmoid output gate.
    Used as the 'full_attention' layer in Qwen3.5 (every 4th layer).
    Qwen3.5-0.8B config: n_heads=8, num_groups=2, head_dim=256, rope_dim=64.
    """

    def __init__(self, idim, n_heads, num_groups, head_dim, rope_dim, dtype,
                 attn_output_gate=True):
        super().__init__()
        self.idim = idim
        self.n_heads = n_heads
        self.num_groups = num_groups
        self.head_dim = head_dim
        # Only the first rope_dim features of each head are rotated (partial RoPE).
        # rope_dim = int(head_dim * partial_rotary_factor) = int(256 * 0.25) = 64
        self.rope_dim = rope_dim
        self.group_size = n_heads // num_groups
        self.n_kv_embed = head_dim * num_groups
        self.odim = n_heads * head_dim
        self.scale = head_dim ** -0.5
        self.has_output_gate = attn_output_gate

        # Q and output gate are tied into a single 2× projection (HF convention).
        # With attn_output_gate=False the gate half is unused; use standard 1× proj.
        if attn_output_gate:
            self.W_query = nn.Linear(idim, self.odim * 2, dtype=dtype, bias=False)
        else:
            self.W_query = nn.Linear(idim, self.odim, dtype=dtype, bias=False)
        self.k_proj = nn.Linear(idim, self.n_kv_embed, dtype=dtype, bias=False)
        self.v_proj = nn.Linear(idim, self.n_kv_embed, dtype=dtype, bias=False)
        self.o_proj = nn.Linear(self.odim, idim, dtype=dtype, bias=False)

        self.q_norm = RMSNorm(head_dim, eps=1e-6)
        self.k_norm = RMSNorm(head_dim, eps=1e-6)

    def forward(self, x, cos, sin, mask=None):
        b, L, _ = x.shape

        # Combined Q + gate projection (2× linear), then split
        q_raw = self.W_query(x)                                  # (B, L, odim[*2])
        if self.has_output_gate:
            q_raw = q_raw.view(b, L, self.n_heads, self.head_dim * 2)
            q, gate = torch.chunk(q_raw, 2, dim=-1)              # each (B, L, H, head_dim)
            gate = gate.reshape(b, L, self.odim)                 # (B, L, odim)
            q = q.transpose(1, 2)                                # (B, H, L, head_dim)
        else:
            q = rearrange(q_raw, 'b l (n d) -> b n l d', n=self.n_heads)
            gate = None

        k = self.k_proj(x)   # (B, L, n_kv_embed)
        v = self.v_proj(x)   # (B, L, n_kv_embed)
        k = rearrange(k, 'b l (g d) -> b g l d', g=self.num_groups)
        v = rearrange(v, 'b l (g d) -> b g l d', g=self.num_groups)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # Partial RoPE: rotate only the leading rope_dim features per head
        q = torch.cat([apply_rope(q[..., :self.rope_dim], cos, sin),
                       q[..., self.rope_dim:]], dim=-1)
        k = torch.cat([apply_rope(k[..., :self.rope_dim], cos, sin),
                       k[..., self.rope_dim:]], dim=-1)

        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        dots = dots.masked_fill(mask, -torch.inf)
        attn = dots.softmax(dim=-1)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h l d -> b l (h d)')

        if self.has_output_gate:
            out = out * torch.sigmoid(gate)

        return self.o_proj(out)

## Gated DeltaNet Linear Attention

In [ ]:
class GatedDeltaNetAttention(nn.Module):
    """
    Gated DeltaNet linear attention, aligned with HF Qwen3_5GatedDeltaNet.

    Recurrent update rule (per step t):
        M[t] = exp(g[t]) * M[t-1]                           # decay old memory
             + k[t]^T * ((v[t] - M[t-1]@k[t]) * beta[t])  # delta correction
        o[t] = scale * q[t] @ M[t]

    Key differences vs. plain DeltaNet:
      - Input-dependent memory decay via Mamba-style discretisation:
            g[t] = softplus(in_proj_a(x[t]) + dt_bias) * (-exp(A_log))
        exp(g[t]) ∈ (0, 1) acts as a per-head forgetting factor each step.
      - Single causal depthwise conv over concatenated [Q, K, V] + SiLU,
        replacing separate per-branch convs.
      - L2-normalised Q and K (no learnable scale) before the recurrence
        (matches use_qk_l2norm_in_kernel=True in the HF kernel call).
      - RMSNormGated output: rms_norm(o) * silu(z), where z = in_proj_z(x).

    Note: replace the Python loop with torch_chunk_gated_delta_rule or the
    FLA triton kernel for training on long sequences.

    Qwen3.5-0.8B config: n_heads=16, key_head_dim=128, value_head_dim=128, conv_kernel_dim=4.
    """

    def __init__(self, idim, n_heads, key_head_dim, value_head_dim, conv_kernel_dim, dtype):
        super().__init__()
        self.n_heads        = n_heads
        self.key_head_dim   = key_head_dim
        self.value_head_dim = value_head_dim
        self.key_dim        = n_heads * key_head_dim    # 16 * 128 = 2048
        self.value_dim      = n_heads * value_head_dim  # 16 * 128 = 2048

        # ── Projections ──────────────────────────────────────────────────────
        # Q + K + V combined; all three pass through the single causal conv
        self.in_proj_qkv = nn.Linear(idim, self.key_dim * 2 + self.value_dim,
                                     dtype=dtype, bias=False)
        # Output gate z; used in RMSNormGated as the silu(z) multiplier
        self.in_proj_z   = nn.Linear(idim, self.value_dim, dtype=dtype, bias=False)
        # Per-head sigmoid beta: learning rate that scales the delta correction
        self.in_proj_b   = nn.Linear(idim, n_heads, dtype=dtype, bias=False)
        # Per-head input for Mamba-style step-size (dt) computation
        self.in_proj_a   = nn.Linear(idim, n_heads, dtype=dtype, bias=False)

        # ── Single causal depthwise conv over [Q, K, V] ──────────────────────
        conv_dim = self.key_dim * 2 + self.value_dim
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=conv_kernel_dim,
                                padding=conv_kernel_dim - 1, groups=conv_dim,
                                dtype=dtype, bias=False)

        # ── Learned decay parameters (Mamba / GLA style) ─────────────────────
        # A_log = log(|A|);  A = -exp(A_log) < 0
        # dt    = softplus(in_proj_a(x) + dt_bias)  →  positive step size
        # g     = dt * A  →  negative log-decay  ⟹  exp(g) ∈ (0, 1)
        A = torch.empty(n_heads).uniform_(0, 16)
        self.A_log   = nn.Parameter(torch.log(A))
        self.dt_bias = nn.Parameter(torch.ones(n_heads, dtype=dtype))

        # ── Output ────────────────────────────────────────────────────────────
        # RMSNorm gated by silu(z), normalises per value-head dimension
        self.norm     = RMSNormGated(value_head_dim, eps=1e-6)
        self.out_proj = nn.Linear(self.value_dim, idim, dtype=dtype, bias=False)

    def forward(self, x, mask=None):
        B, L, _ = x.shape

        # 1. Combined QKV projection → causal depthwise conv → SiLU
        qkv = self.in_proj_qkv(x)                          # (B, L, 2*key_dim + value_dim)
        qkv = self.conv1d(qkv.transpose(1, 2))[..., :L]   # (B, conv_dim, L) – discard right pad
        qkv = F.silu(qkv).transpose(1, 2)                 # (B, L, 2*key_dim + value_dim)

        q, k, v = torch.split(qkv, [self.key_dim, self.key_dim, self.value_dim], dim=-1)

        # 2. Output gate, beta (learning rate), and input-dependent decay g
        z    = self.in_proj_z(x)                           # (B, L, value_dim)
        beta = torch.sigmoid(self.in_proj_b(x))            # (B, L, H)
        A    = -self.A_log.float().exp()                   # (H,) negative log-decay magnitude
        dt   = F.softplus(self.in_proj_a(x).float() +
                          self.dt_bias.float())             # (B, L, H) positive step sizes
        g    = dt * A                                      # (B, L, H) negative log-decay

        # 3. Reshape to (B, L, H, D) then L2-normalise Q and K (no learnable scale)
        q = l2norm(q.view(B, L, self.n_heads, self.key_head_dim))
        k = l2norm(k.view(B, L, self.n_heads, self.key_head_dim))
        v = v.view(B, L, self.n_heads, self.value_head_dim)

        # 4. Transpose to (B, H, L, D) and cast to float32 for the recurrence
        q    = q.transpose(1, 2).float()                   # (B, H, L, dk)
        k    = k.transpose(1, 2).float()                   # (B, H, L, dk)
        v    = v.transpose(1, 2).float()                   # (B, H, L, dv)
        beta = beta.transpose(1, 2)                        # (B, H, L)
        g    = g.transpose(1, 2)                           # (B, H, L)

        scale = self.key_head_dim ** -0.5

        # 5. Recurrent gated delta-rule (float32 throughout for numerical stability)
        out = torch.zeros(B, self.n_heads, L, self.value_head_dim,
                          device=x.device, dtype=torch.float32)
        M   = torch.zeros(B, self.n_heads, self.key_head_dim, self.value_head_dim,
                          device=x.device, dtype=torch.float32)

        for t in range(L):
            q_t   = q[:, :, t] * scale                          # (B, H, dk)
            k_t   = k[:, :, t]                                  # (B, H, dk)
            v_t   = v[:, :, t]                                  # (B, H, dv)
            b_t   = beta[:, :, t, None].float()                 # (B, H, 1)
            # exp(g) ∈ (0, 1): fraction of memory retained this step
            decay = g[:, :, t].float().exp()[..., None, None]   # (B, H, 1, 1)

            M     = M * decay                                    # forget old memory
            Mk    = einsum('bhd, bhdv -> bhv', k_t, M)          # memory prediction
            M     = M + einsum('bhd, bhv -> bhdv',
                               k_t, (v_t - Mk) * b_t)           # delta update
            out[:, :, t] = einsum('bhd, bhdv -> bhv', q_t, M)  # read out

        # 6. RMSNormGated (norm then silu-gate) then project to model dim
        out = out.to(x.dtype).transpose(1, 2)              # (B, L, H, dv)
        z   = z.view(B, L, self.n_heads, self.value_head_dim)
        out = self.norm(out, z)                            # (B, L, H, dv)
        out = out.reshape(B, L, self.value_dim)
        return self.out_proj(out)

## TokenizerSetup

In [5]:
import re
from pathlib import Path
from tokenizers import Tokenizer


class Qwen3_5Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(
        self,
        tokenizer_file_path="tokenizer.json",
        repo_id=None,
        apply_chat_template=True,
        add_generation_prompt=False,
        add_thinking=False,
    ):
        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        # Mirrors Qwen3.5 chat_template behavior:
        # add_generation_prompt + thinking => "<think>\n"
        # add_generation_prompt + no thinking => empty think scaffold
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant\n"
            if self.add_thinking:
                s += "<think>\n"
            else:
                s += "<think>\n\n</think>\n\n"
        return s

## Qwen 3.5 Building Blocks

In [6]:
import torch
from torch import nn
from torch.nn import functional as F


class GatedFeedForward(nn.Module):
    def __init__(self, idim, hidden_dim, dtype):
        super().__init__()
        self.gate_proj = nn.Linear(idim, hidden_dim, dtype=dtype, bias=False)
        self.up_proj   = nn.Linear(idim, hidden_dim, dtype=dtype, bias=False)
        self.down_proj = nn.Linear(hidden_dim, idim,  dtype=dtype, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class GatedAttentionBlock(nn.Module):
    """
    Full (quadratic-softmax) GQA block with sigmoid output gate.
    Instantiated every 4th layer (full_attention_interval=4).
    Qwen3.5-0.8B: n_heads=8, num_groups=2, head_dim=256, rope_dim=64.
    """
    def __init__(self, dim, n_heads, num_groups, head_dim, rope_dim, mlp_dim, dtype):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = GQAttention(dim, n_heads=n_heads, num_groups=num_groups,
                                 head_dim=head_dim, rope_dim=rope_dim,
                                 dtype=dtype, attn_output_gate=True)
        self.norm2 = RMSNorm(dim)
        self.ff    = GatedFeedForward(dim, mlp_dim, dtype)

    def forward(self, x, cos, sin, mask=None):
        x = self.attn(self.norm1(x), cos, sin, mask) + x
        x = self.ff(self.norm2(x)) + x
        return x


class GatedDeltaNetBlock(nn.Module):
    """
    Linear-attention DeltaNet block with SiLU output gate.
    Occupies the 3 layers preceding each GatedAttentionBlock.
    Qwen3.5-0.8B: n_heads=16, key_head_dim=128, value_head_dim=128, conv_kernel_dim=4.
    """
    def __init__(self, dim, n_heads, key_head_dim, value_head_dim,
                 conv_kernel_dim, mlp_dim, dtype):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = GatedDeltaNetAttention(dim, n_heads=n_heads,
                                            key_head_dim=key_head_dim,
                                            value_head_dim=value_head_dim,
                                            conv_kernel_dim=conv_kernel_dim,
                                            dtype=dtype)
        self.norm2 = RMSNorm(dim)
        self.ff    = GatedFeedForward(dim, mlp_dim, dtype)

    def forward(self, x, cos=None, sin=None, mask=None):
        # DeltaNet does not use positional encoding; cos/sin kept for a
        # uniform block interface so the model loop can call all blocks alike.
        x = self.attn(self.norm1(x)) + x
        x = self.ff(self.norm2(x)) + x
        return x


class Qwen35Model(nn.Module):
    """
    Qwen3.5 language model backbone.

    Architecture: 6 × (3 × GatedDeltaNet → 1 × GatedAttention) = 24 layers total.
    layer_types list from config drives which block class is used per layer.

    Key differences from Qwen3:
      - Hybrid linear / full attention (3:1 ratio)
      - Gated DeltaNet (delta-rule linear attention) for 18 layers
      - Gated full GQA (softmax) for 6 layers
      - Partial RoPE on full-attention heads (rope_dim = head_dim * 0.25 = 64)
      - Sigmoid output gate on full-attention; SiLU gate on DeltaNet
      - Larger vocab: 248 320
    """

    def __init__(self, dim, depth, n_heads, num_groups, head_dim, rope_dim,
                 linear_n_heads, linear_key_head_dim, linear_value_head_dim,
                 linear_conv_kernel_dim, mlp_dim, vocab_size, context_length,
                 layer_types, dtype=torch.bfloat16):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim, dtype=dtype)

        self.blocks = nn.ModuleList()
        for ltype in layer_types:
            if ltype == 'full_attention':
                self.blocks.append(
                    GatedAttentionBlock(dim, n_heads, num_groups, head_dim,
                                        rope_dim, mlp_dim, dtype))
            else:  # 'linear_attention'
                self.blocks.append(
                    GatedDeltaNetBlock(dim, linear_n_heads, linear_key_head_dim,
                                       linear_value_head_dim, linear_conv_kernel_dim,
                                       mlp_dim, dtype))

        self.final_norm = RMSNorm(dim, eps=1e-6)
        # Weight-tied to tok_emb (tie_word_embeddings=true in config)
        self.final_proj = nn.Linear(dim, vocab_size, bias=False, dtype=dtype)

        # RoPE tables sized for rope_dim only (partial_rotary_factor=0.25 → 64 dims).
        # theta=10_000_000 per Qwen3.5 config (vs 1_000_000 in Qwen3).
        cos, sin = rope_rotate(rope_dim, context_length, theta=10_000_000)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        self.dtype = dtype

    def forward(self, inp):
        emb = self.tok_emb(inp)
        n   = emb.shape[1]
        mask = torch.triu(torch.ones(n, n, device=inp.device, dtype=torch.bool), diagonal=1)
        x = emb

        for block in self.blocks:
            x = block(x, self.cos, self.sin, mask)

        x = self.final_norm(x)
        x = self.final_proj(x.to(self.dtype))
        return x



In [ ]:
QWEN3_5_CONFIG = {
    "vocab_size":              248_320,
    "context_length":          262_144,
    "emb_dim":                 1024,
    # Full attention (GQA) params
    "n_heads":                 8,
    "n_kv_groups":             2,
    "head_dim":                256,
    "rope_dim":                64,       # head_dim * partial_rotary_factor (0.25)
    # Linear attention (DeltaNet) params
    "linear_n_heads":          16,
    "linear_key_head_dim":     128,
    "linear_value_head_dim":   128,
    "linear_conv_kernel_dim":  4,
    # Shared
    "n_layers":                24,
    "hidden_dim":              3584,
    "layer_types": (["linear_attention"] * 3 + ["full_attention"]) * 6,
    "dtype": torch.bfloat16,
}

model = Qwen35Model(
    dim=QWEN3_5_CONFIG["emb_dim"],
    depth=QWEN3_5_CONFIG["n_layers"],
    n_heads=QWEN3_5_CONFIG["n_heads"],
    num_groups=QWEN3_5_CONFIG["n_kv_groups"],
    head_dim=QWEN3_5_CONFIG["head_dim"],
    rope_dim=QWEN3_5_CONFIG["rope_dim"],
    linear_n_heads=QWEN3_5_CONFIG["linear_n_heads"],
    linear_key_head_dim=QWEN3_5_CONFIG["linear_key_head_dim"],
    linear_value_head_dim=QWEN3_5_CONFIG["linear_value_head_dim"],
    linear_conv_kernel_dim=QWEN3_5_CONFIG["linear_conv_kernel_dim"],
    mlp_dim=QWEN3_5_CONFIG["hidden_dim"],
    vocab_size=QWEN3_5_CONFIG["vocab_size"],
    context_length=QWEN3_5_CONFIG["context_length"],
    layer_types=QWEN3_5_CONFIG["layer_types"],
    dtype=QWEN3_5_CONFIG["dtype"],
)
out = model(torch.tensor([1, 2, 3]).unsqueeze(0))

print("Model output shape : ", out.shape)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

# print("\nModel : \n", model)

print(f"float32 (PyTorch default): {model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

## Decoding Algorithms

In [7]:
import torch
import torch.nn.functional as F


def greedy_decoding(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(token_ids)[:, -1]
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            if (eos_token_id is not None
                    and torch.all(next_token == eos_token_id)):
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)


def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0, recent_tokens=None):
    """
    Apply temperature, top-k, top-p, and repetition penalty sampling.
    """

    # Temperature scaling
    if temperature != 1.0:
        logits = logits / temperature

    # Repetition penalty (penalize tokens that appeared recently)
    if repetition_penalty != 1.0 and recent_tokens is not None:
        for token in set(recent_tokens.tolist()):
            logits[..., token] /= repetition_penalty

    # Top-k filtering
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        values, indices = torch.topk(logits, top_k)
        mask = logits < values[..., -1, None]
        logits = logits.masked_fill(mask, float("-inf"))

    # Top-p (nucleus) filtering
    if top_p is not None:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(probs, dim=-1)

        # Mask tokens where cumulative prob > top_p
        sorted_mask = cumulative_probs > top_p
        # shift mask so at least 1 token is kept
        sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
        sorted_mask[..., 0] = False

        # apply mask
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        logits = logits.masked_fill(mask, float("-inf"))

    # Convert to probabilities
    probs = F.softmax(logits, dim=-1)

    # Sample
    next_token = torch.multinomial(probs, num_samples=1)
    return next_token

def advance_decoding(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=1.0,
    top_k=None,
    top_p=None,
    repetition_penalty=1.0,
    window_size=50,
):
    """
    Streaming text generation with advanced sampling.
    """
    model.eval()
    recent_tokens = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(token_ids)[:, -1, :]   # logits of last token
            next_token = sample_next_token(
                out,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                recent_tokens=token_ids[:, -window_size:].flatten()
            )

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            # Update sequence
            token_ids = torch.cat([token_ids, next_token], dim=1)
            recent_tokens.append(next_token.item())

## Load Weight via Named Parameters Mapping

In [9]:
import torch


def load_weights_into_qwen3_5(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    if "model.embed_tokens.weight" in params:
        model_prefix = "model"
    elif "model.language_model.embed_tokens.weight" in params:
        model_prefix = "model.language_model"
    else:
        raise KeyError("Could not find embed token weights in checkpoint.")

    def pkey(suffix):
        return f"{model_prefix}.{suffix}"

    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params[pkey("embed_tokens.weight")],
        pkey("embed_tokens.weight"),
    )

    n_layers = param_config["n_layers"]
    layer_types = param_config.get("layer_types", ["full_attention"] * n_layers)

    for l in range(n_layers):
        block = model.blocks[l]
        layer_type = layer_types[l]

        if layer_type == "full_attention":
            att = block.attn
            att.W_query.weight = assign(
                att.W_query.weight,
                params[pkey(f"layers.{l}.self_attn.q_proj.weight")],
                pkey(f"layers.{l}.self_attn.q_proj.weight"),
            )
            att.k_proj.weight = assign(
                att.k_proj.weight,
                params[pkey(f"layers.{l}.self_attn.k_proj.weight")],
                pkey(f"layers.{l}.self_attn.k_proj.weight"),
            )
            att.v_proj.weight = assign(
                att.v_proj.weight,
                params[pkey(f"layers.{l}.self_attn.v_proj.weight")],
                pkey(f"layers.{l}.self_attn.v_proj.weight"),
            )
            att.o_proj.weight = assign(
                att.o_proj.weight,
                params[pkey(f"layers.{l}.self_attn.o_proj.weight")],
                pkey(f"layers.{l}.self_attn.o_proj.weight"),
            )
            if hasattr(att, "q_norm") and att.q_norm is not None:
                att.q_norm.weight = assign(
                    att.q_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.q_norm.weight")],
                    pkey(f"layers.{l}.self_attn.q_norm.weight"),
                )
            if hasattr(att, "k_norm") and att.k_norm is not None:
                att.k_norm.weight = assign(
                    att.k_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.k_norm.weight")],
                    pkey(f"layers.{l}.self_attn.k_norm.weight"),
                )

        elif layer_type == "linear_attention":
            lat = block.attn
            lat.dt_bias = assign(
                lat.dt_bias,
                params[pkey(f"layers.{l}.linear_attn.dt_bias")],
                pkey(f"layers.{l}.linear_attn.dt_bias"),
            )
            lat.A_log = assign(
                lat.A_log,
                params[pkey(f"layers.{l}.linear_attn.A_log")],
                pkey(f"layers.{l}.linear_attn.A_log"),
            )
            lat.conv1d.weight = assign(
                lat.conv1d.weight,
                params[pkey(f"layers.{l}.linear_attn.conv1d.weight")],
                pkey(f"layers.{l}.linear_attn.conv1d.weight"),
            )
            lat.norm.weight = assign(
                lat.norm.weight,
                params[pkey(f"layers.{l}.linear_attn.norm.weight")],
                pkey(f"layers.{l}.linear_attn.norm.weight"),
            )
            lat.out_proj.weight = assign(
                lat.out_proj.weight,
                params[pkey(f"layers.{l}.linear_attn.out_proj.weight")],
                pkey(f"layers.{l}.linear_attn.out_proj.weight"),
            )
            lat.in_proj_qkv.weight = assign(
                lat.in_proj_qkv.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight"),
            )
            lat.in_proj_z.weight = assign(
                lat.in_proj_z.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_z.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_z.weight"),
            )
            lat.in_proj_b.weight = assign(
                lat.in_proj_b.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_b.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_b.weight"),
            )
            lat.in_proj_a.weight = assign(
                lat.in_proj_a.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_a.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_a.weight"),
            )

        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        block.norm1.weight = assign(
            block.norm1.weight,
            params[pkey(f"layers.{l}.input_layernorm.weight")],
            pkey(f"layers.{l}.input_layernorm.weight"),
        )

        block.ff.gate_proj.weight = assign(
            block.ff.gate_proj.weight,
            params[pkey(f"layers.{l}.mlp.gate_proj.weight")],
            pkey(f"layers.{l}.mlp.gate_proj.weight"),
        )
        block.ff.up_proj.weight = assign(
            block.ff.up_proj.weight,
            params[pkey(f"layers.{l}.mlp.up_proj.weight")],
            pkey(f"layers.{l}.mlp.up_proj.weight"),
        )
        block.ff.down_proj.weight = assign(
            block.ff.down_proj.weight,
            params[pkey(f"layers.{l}.mlp.down_proj.weight")],
            pkey(f"layers.{l}.mlp.down_proj.weight"),
        )
        block.norm2.weight = assign(
            block.norm2.weight,
            params[pkey(f"layers.{l}.post_attention_layernorm.weight")],
            pkey(f"layers.{l}.post_attention_layernorm.weight"),
        )

    model.final_norm.weight = assign(
        model.final_norm.weight,
        params[pkey("norm.weight")],
        pkey("norm.weight"),
    )

    if "lm_head.weight" in params:
        model.final_proj.weight = assign(model.final_proj.weight, params["lm_head.weight"], "lm_head.weight")
    elif pkey("lm_head.weight") in params:
        model.final_proj.weight = assign(model.final_proj.weight, params[pkey("lm_head.weight")], pkey("lm_head.weight"))
    else:
        model.final_proj.weight = model.tok_emb.weight
        print("Model uses weight tying.")

## Testing

In [10]:
import torch
import os, json
from huggingface_hub import snapshot_download

from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download


def test_qwen3_5_0_8B(prompt, config):

    layer_types = (["linear_attention"] * 3 + ["full_attention"]) * 6  # 24 layers
    config["layer_types"] = layer_types

    model = Qwen35Model(
        dim=config["emb_dim"],
        depth=config["n_layers"],
        n_heads=config["n_heads"],
        num_groups=config["n_kv_groups"],
        head_dim=config["head_dim"],
        rope_dim=config["rope_dim"],
        linear_n_heads=config["linear_n_heads"],
        linear_key_head_dim=config["linear_key_head_dim"],
        linear_value_head_dim=config["linear_value_head_dim"],
        linear_conv_kernel_dim=config["linear_conv_kernel_dim"],
        mlp_dim=config["hidden_dim"],
        vocab_size=config["vocab_size"],
        context_length=config["context_length"],
        layer_types=layer_types,
        dtype=config["dtype"],
    )
    device = torch.device("cuda")



    repo_id = "Qwen/Qwen3.5-0.8B"
    local_dir = Path(repo_id).parts[-1]

    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in sorted(set(index["weight_map"].values())):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)


    load_weights_into_qwen3_5(model, config, weights_dict)
    model.to(device)
    del weights_dict

    tokenizer_file_path = "Qwen3.5-0.8B/tokenizer.json"

    hf_hub_download(
        repo_id=repo_id,
        filename="tokenizer.json",
        local_dir=local_dir,
    )

    tokenizer = Qwen3_5Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )


    print(f"Prompt : {prompt}")
    input_token_ids = tokenizer.encode(prompt)
    text = tokenizer.decode(input_token_ids)
    print(f"Decoded Text: {text}")

    input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

    for token in advance_decoding(
            model=model,
            token_ids=input_token_ids_tensor,
            max_new_tokens=8192,
            eos_token_id=tokenizer.eos_token_id,
            temperature=0.9,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.2,
            window_size=50
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )

In [ ]:
prompt = "Please explain the climate change and how it impacts our future."

QWEN3_5_CONFIG = {
    # Qwen3.5-0.8B text_config values
    "vocab_size":             248_320,
    "context_length":         262_144,
    "emb_dim":                1024,
    "n_layers":               24,
    "hidden_dim":             3584,
    # Full attention (GQA) — every 4th layer
    "n_heads":                8,
    "n_kv_groups":            2,
    "head_dim":               256,
    "rope_dim":               64,      # head_dim * partial_rotary_factor (0.25)
    # Linear attention (GatedDeltaNet) — remaining 3/4 layers
    "linear_n_heads":         16,
    "linear_key_head_dim":    128,
    "linear_value_head_dim":  128,
    "linear_conv_kernel_dim": 4,
    "dtype": torch.bfloat16,
}
test_qwen3_5_0_8B(prompt, QWEN3_5_CONFIG)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model uses weight tying.
Prompt : Please explain the climate change and how it impacts our future.
Decoded Text: <|im_start|>user
Please explain the climate change and how it impacts our future.<|im_end|>
<|im_start|>assistant
<think>

Here's a thinking process that leads to the suggested explanation:

1.  **Deconstruct the Request:**
    *   **Topic:** Climate Change (the phenomenon itself).
    *   **Specific Aspects:** Explain how it impacts our future (consequences, effects on society/individuals/environment).
    *   **Goal:** Provide a clear, informative, and engaging explanation.

2.  **Initial Brainstorming & Structuring:**
    *   What is climate change? (Scientific consensus: human-caused, rising temperatures).
    *   Key driver emissions/energy consumption/greenhouse gases.
    *   Direct impacts on physical planet's weather patterns (climate zones).
    *   Impact on agriculture/food security.
    *   Health and safety risks (heatwaves, disease outbreaks).
    *   Social/e